# EDA and model experiments

Student: Vaishnavi Nirpal (CS3234)

This notebook explores the **IBM HR Analytics Employee Attrition & Performance** sample and points to the official training script `python -m src.train_model`.

The CSV is fictional IBM sample data, not real employees.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

import config
from src.data_loader import load_dataset, profile_dataset
from src.visualizations import (
    plot_attrition_rate_by,
    plot_class_distribution,
    plot_correlation_heatmap,
    plot_numeric_distributions,
    plot_years_at_company,
)

pd.set_option("display.max_columns", 50)
df = load_dataset(download_if_missing=True)
profile = profile_dataset(df)
profile

## Dataset overview

Run the cell above first. It prints the real row count, column count, dtypes, missing values, duplicates, and class balance from the CSV on disk.

In [ ]:
display(df.head())
print("Missing cells:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print(df.dtypes)
df[config.TARGET_COLUMN].value_counts()

In [ ]:
fig = plot_class_distribution(df)
plt.show()

for column, title in [
    ("OverTime", "Attrition by overtime"),
    ("JobRole", "Attrition by job role"),
    ("Department", "Attrition by department"),
    ("JobSatisfaction", "Attrition by job satisfaction"),
]:
    if column in df.columns:
        fig = plot_attrition_rate_by(df, column, title)
        plt.show()

fig = plot_years_at_company(df)
if fig:
    plt.show()

numeric_cols = ["Age", "MonthlyIncome", "DistanceFromHome", "YearsAtCompany", "TotalWorkingYears", "PercentSalaryHike"]
fig = plot_numeric_distributions(df, numeric_cols)
if fig:
    plt.show()

fig = plot_correlation_heatmap(df)
if fig:
    plt.show()

## Modelling

Do not retrain ad-hoc in this notebook if you need the production pipeline. The comparable, leakage-safe experiment lives in `src/train_model.py`.

After training, load `reports/model_comparison.csv` and `reports/training_summary.json` for the measured scores.

In [ ]:
comparison_path = ROOT / "reports" / "model_comparison.csv"
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
else:
    print("Train first: python -m src.train_model")